# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [ ]:
# 在此写下你的答案：
# 1. 每条记录代表一名电商用户（CustomerID），包含其行为、偏好与流失标签
# 2. 项目的目标变量是 Churn（流失），0=未流失，1=已流失
# 3. CustomerID 是用户标识符，不是业务指标；求平均、求和等操作无业务意义，若作为数值特征参与建模，会导致模型把 ID 数值大小误认为业务规律


---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [ ]:
def build_quality_report(data):
    """返回字段级数据质量报告。"""
    return pd.DataFrame({
        "数据类型": data.dtypes,
        "缺失数量": data.isna().sum(),
        "缺失比例(%)": (data.isna().mean() * 100).round(2),
        "唯一值数量": data.nunique()
    })

quality_before = build_quality_report(raw_df)
display(quality_before)


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [ ]:
print("完全重复行数：", raw_df.duplicated().sum())
print("CustomerID 重复数量：", raw_df.duplicated(subset=["CustomerID"]).sum())
print("\nChurn 频数：")
print(raw_df["Churn"].value_counts())
print(f"\n流失率：{raw_df['Churn'].value_counts().get(1, 0) / len(raw_df) * 100:.2f}%")

for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"\n{col}")
    print(raw_df[col].value_counts())


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [ ]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [ ]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    df = data.copy()
    logs = []

    rows_before = len(df)
    df = df.drop_duplicates()
    logs.append({"step": "drop_duplicates", "before": rows_before, "after": len(df), "affected": rows_before - len(df)})

    for col in NUMERIC_MISSING_COLS:
        missing = df[col].isna().sum()
        if missing > 0:
            med = df[col].median()
            df[col] = df[col].fillna(med)
            logs.append({"step": f"fillna({col})", "before": len(df), "after": len(df), "affected": missing, "value": med})

    for col, mapping in CATEGORY_MAPPINGS.items():
        for old, new in mapping.items():
            cnt = (df[col] == old).sum()
            if cnt > 0:
                df[col] = df[col].replace({old: new})
                logs.append({"step": f"replace({col})", "before": len(df), "after": len(df), "affected": cnt, "rule": f"{old} -> {new}"})

    df["Churn"] = df["Churn"].astype(int)
    df["Complain"] = df["Complain"].astype(int)
    return df, pd.DataFrame(logs)


### 任务 3：运行清洗函数并查看日志

In [ ]:
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

display(cleaning_log)
cleaned_df.head()


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [ ]:
def iqr_outlier_summary(series):
    """返回数值序列的IQR候选异常值摘要。"""
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((s < lower) | (s > upper)).sum()
    })

In [ ]:
tenure_bins = [0, 5, 10, 15, 20, 25, 30, float("inf")]
tenure_labels = ["0-5年", "5-10年", "10-15年", "15-20年", "20-25年", "25-30年", "30年以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=False)

cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

print("\n=== TenureGroup 分布 ===")
print(cleaned_df["TenureGroup"].value_counts().sort_index())

outlier_report = pd.DataFrame([
    iqr_outlier_summary(cleaned_df["WarehouseToHome"]),
    iqr_outlier_summary(cleaned_df["OrderCount"]),
    iqr_outlier_summary(cleaned_df["CashbackAmount"]),
], index=["WarehouseToHome", "OrderCount", "CashbackAmount"])
print("\n=== 候选异常值报告 ===")
display(outlier_report)


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [ ]:
business_rule_report = pd.DataFrame({
    "规则": ["使用时长小于 0", "仓库距离小于 0", "订单数小于或等于 0", "返现金额小于 0"],
    "不合规记录数": [
        int((cleaned_df["HourSpendOnApp"] < 0).sum()),
        int((cleaned_df["WarehouseToHome"] < 0).sum()),
        int((cleaned_df["OrderCount"] <= 0).sum()),
        int((cleaned_df["CashbackAmount"] < 0).sum()),
    ]
})
display(business_rule_report)

# 处理结论：若不合规记录数为 0，说明数据在业务规则层面未发现明显问题；若有不合规记录，需进一步排查数据来源。
# business_rule_report = pd.DataFrame({
#     "规则": [...],
#     "不合规记录数": [...]
# })
# display(business_rule_report)
#
# 处理结论：

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [ ]:
# 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)
print("=== 清洗后质量报告 ===")
display(quality_after)

# 导出交付物
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

print("\n=== 验收检查 ===")
print("输出目录：", OUTPUT_DIR.resolve())
print("文件列表：")
for f in OUTPUT_DIR.iterdir():
    print(f"  {f.name}: {f.stat().st_size:,} bytes")
print("\n清洗后行数：", len(cleaned_df))
print("清洗后列数：", cleaned_df.shape[1])
print("缺失值总计：", cleaned_df.isna().sum().sum())

## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

# 复盘答案：
# 1. 发现数值字段缺失（Tenure等7列）、类别同义异名（Phone/Mobile Phone等）、候选异常值。
# 2. 缺失值用中位数填补（不按Churn分组）；类别不一致通过映射表统一；候选异常值只记录不删除。
# 3. 清洗后数据无缺失、类别统一、新增TenureGroup和IsMobileLogin特征，可直接用于分组分析和可视化。
# 4. 候选异常值的取舍、TenureGroup分箱阈值选择，都需业务人员确认。
